# PART 03 · 2장 작업형 2 데이터 전처리 — 연습 문제 정답

교재 237쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [ ]:
import pandas as pd

base = 'data/part03_type2/regression_delivery_minutes'
train = pd.read_csv(base + '/train.csv')
test = pd.read_csv(base + '/test.csv')
print(train.shape, test.shape)

(840, 15) (280, 14)


## 문제 01

회귀 target인 delivery_minutes에서 0은 실제 배달 시간이 아니라 미기록값을 표시한 값이라고 한다. target이 NaN이거나 0인 행을 제외한 뒤, order_id를 입력 데이터에서 제거하고 X와 y를 분리하는 코드를 작성하시오.

**정답 코드**

In [ ]:
target = 'delivery_minutes'
valid_cond = (train[target].notna() & (train[target] != 0))
train_valid = train.loc[valid_cond].copy()
y = train_valid[target]
X = train_valid.drop(columns=[target, 'order_id'])

print('before:', train.shape[0])
print('after:', train_valid.shape[0])
print('X:', X.shape)
print('y:', y.shape)

before: 840
after: 840
X: (840, 13)
y: (840,)


**간단 해설**

target은 모델이 맞혀야 할 정답이므로 비어 있거나 유효하지 않은 값을 임의로 채우지 않는다. 유효한 행을 먼저 남긴 뒤 target과 식별자를 입력 데이터에서 제외한다.

## 문제 02

다음 코드에서 fill_value=0의 역할을 설명하시오.

```python
X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
```

**정답 코드**

In [ ]:
# 확인 코드: X_train에만 있는 범주가 있을 때 어떤 일이 생기는지 작은 예로 확인한다.
train_sample = train[['store_type', 'distance_km']].head(50)
test_sample = train_sample[train_sample['store_type'] != '피자']

X_train_encoded = pd.get_dummies(train_sample, dtype=int)
X_test_encoded = pd.get_dummies(test_sample, dtype=int)

print('train 더미 컬럼:', X_train_encoded.columns.tolist())
print('test 더미 컬럼(정렬 전):', X_test_encoded.columns.tolist())

X_test_encoded = X_test_encoded.reindex(
    columns=X_train_encoded.columns,
    fill_value=0
)
print('test 더미 컬럼(정렬 후):', X_test_encoded.columns.tolist())
print(
    '컬럼 순서 일치:',
    list(X_test_encoded.columns) == list(X_train_encoded.columns)
)
print(X_test_encoded.head(3))

train 더미 컬럼: ['distance_km', 'store_type_도시락', 'store_type_분식', 'store_type_샐러드', 'store_type_치킨', 'store_type_카페', 'store_type_피자']
test 더미 컬럼(정렬 전): ['distance_km', 'store_type_도시락', 'store_type_분식', 'store_type_샐러드', 'store_type_치킨', 'store_type_카페']
test 더미 컬럼(정렬 후): ['distance_km', 'store_type_도시락', 'store_type_분식', 'store_type_샐러드', 'store_type_치킨', 'store_type_카페', 'store_type_피자']
컬럼 순서 일치: True
   distance_km  store_type_도시락  store_type_분식  store_type_샐러드  store_type_치킨  \
0         4.39               0              0               0              1   
1         3.91               0              1               0              0   
2         1.52               0              0               0              0   

   store_type_카페  store_type_피자  
0              0              0  
1              0              0  
2              1              0  


**간단 해설**

검증 데이터와 test 데이터를 X_train의 더미 컬럼 목록에 맞춘다. X_train에는 있지만 해당 데이터에 없는 범주 컬럼은 0으로 채워 입력 컬럼 수와 순서를 같게 만든다.

---

**정답** 1. 유효한 target 행을 먼저 필터링한 뒤 X와 y를 분리한다, 2. test에 없는 X 기준 컬럼을 0으로 채운다.